# 09 - Module 4 validation: ML baseline vs LLM agents

Trains a Random Forest baseline on the exact same features provided to
the LLM agents (raw features plus Module 1-3 quality scores in C3) and
compares the ML decisions with each LLM via Cohen's kappa. A kappa close
to 1 supports the use of LLM agents as decision-quality proxies.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.metrics.llm import cohen_kappa_pair, decision_accuracy
from xai_blockchain_framework.models import train_random_forest
from xai_blockchain_framework.utils import save_csv

set_seed()

raw = pd.read_csv(PATHS.results_dir / 'module4_llm_raw_responses.csv')
raw = raw.dropna(subset=['Decision']).copy()


## Build feature vectors per condition

In [ ]:
splits = {
    'Elliptic': np.load(PATHS.data_processed / 'elliptic_splits.npz'),
    'Ethereum': np.load(PATHS.data_processed / 'ethereum_splits.npz'),
}

# Cache one scaler + one RF classifier per dataset (not per condition).
trained: dict[str, tuple] = {}
for ds, sp in splits.items():
    scaler = StandardScaler().fit(sp['X_train'])
    X_tr_s = scaler.transform(sp['X_train'])
    clf = train_random_forest(X_tr_s, sp['y_train'])
    trained[ds] = (scaler, clf)

# Group by (Dataset, BRAS_label, Condition) to keep each evaluation cohort
# separate; the same TX_idx can appear across multiple BRAS_label groups
# (e.g. high-BRAS vs low-BRAS configurations), which is why we must NOT
# group only by (Dataset, Condition).
comparison_rows = []
for (ds, bras_label, cond), grp in raw.groupby(['Dataset', 'BRAS_label', 'Condition']):
    X_test = splits[ds]['X_test']
    y_test = splits[ds]['y_test']
    scaler, clf = trained[ds]

    # For a given (dataset, BRAS_label, condition) the same transaction is
    # scored by three agents. Pivot to get one row per TX_idx with the
    # three agent decisions as columns.
    pivoted = grp.pivot_table(
        index='TX_idx',
        columns='Agent',
        values='Decision',
        aggfunc='first',
    )
    tx_indices = pivoted.index.to_numpy(dtype=int)
    X_sub_s = scaler.transform(X_test[tx_indices])
    ml_pred = clf.predict(X_sub_s)
    ml_labels = np.array(['fraud' if p == 1 else 'legitimate' for p in ml_pred])
    y_sub = y_test[tx_indices]

    for agent in pivoted.columns:
        decisions = pivoted[agent].to_numpy()
        mask = pd.notna(decisions)
        if mask.sum() < 2:
            continue
        llm_dec = decisions[mask].tolist()
        ml_dec = ml_labels[mask].tolist()
        y_m = y_sub[mask]
        comparison_rows.append({
            'Dataset': ds,
            'BRAS_label': bras_label,
            'Condition': cond,
            'Agent': agent,
            'DA_ML': decision_accuracy(ml_dec, y_m),
            'DA_LLM': decision_accuracy(llm_dec, y_m),
            'Kappa_ML_LLM': cohen_kappa_pair(ml_dec, llm_dec),
            'N': int(mask.sum()),
        })

df = pd.DataFrame(comparison_rows)
print(df.to_string(index=False))
save_csv(df, PATHS.results_dir / 'module4_ml_baseline_results.csv')
